In [1]:
!pip install janome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 44.1 MB/s  0:00:00m0:00:010:01

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import glob
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# ★ Janomeのインポート
from janome.tokenizer import Tokenizer

# --- Janomeの初期化 ---
tokenizer_obj = Tokenizer()

# Janomeの品詞大分類に合わせたルール設定
FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞", "名詞,代名詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "副詞", "感動詞", "形状詞"}

def analyze_text_files(folder_path):
    """指定されたフォルダ内のテキストファイルを読み込み、Janomeで形態素解析と基本特徴量の抽出を行う"""
    
    # -----------------------------------------------------------------
    # 【ファイルの指定】
    # -----------------------------------------------------------------
    # Jupyterの実行位置（現在のフォルダ）を取得
    current_dir = os.getcwd()

    # ▼ パターンA: 同じ場所にある「Human_text_01.txt」を直接処理する場合 (現在はこっち)
    filepaths = [os.path.join(current_dir, "Human_text_01.txt")]
    
    # ▼ パターンB: もし将来「Human_text」というフォルダに30個をまとめた場合（戻す時は下の3行の#を消す）
    # base_folder = os.path.join(current_dir, "Human_text")
    # filepaths = sorted(
    #     glob.glob(os.path.join(base_folder, "*.txt")),
    #     key=lambda x: int(re.search(r"\d+", os.path.basename(x)).group()),
    # )
    # -----------------------------------------------------------------

    files = []
    tokenized_texts = []
    content_texts = []
    function_texts = []
    function_counts = []
    content_counts = []
    function_pers = []
    content_pers = []
    feature_data = []

    for filepath in filepaths:
        if not os.path.exists(filepath):
            print(f"【エラー】ファイルが見つかりません: {filepath}")
            continue

        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
            files.append((filepath, text))

        pos_counter = Counter()
        particle_counter = Counter()
        tokens, content_words, function_words, all_words = [], [], [], []
        word_count = content_count = function_count = 0

        # 記号のカウントと文長計算
        comma_count = text.count("、")
        period_count = text.count("。")
        exclamation_count = text.count("！")
        question_count = text.count("？")

        sentences = re.split(r"[。！？]", text)
        sentence_lengths = [len(s.strip()) for s in sentences if s.strip()]
        avg_sentence_length = np.mean(sentence_lengths) if sentence_lengths else 0

        # --- Janomeによる形態素解析 ---
        tokens_janome = tokenizer_obj.tokenize(text)
        
        for word in tokens_janome:
            # word.part_of_speech は '名詞,一般,*,*' のようなカンマ区切りの文字列
            pos_full = word.part_of_speech
            pos = pos_full.split(',')[0]  # 大分類（例: '名詞'）
            surface = word.surface
            base = word.base_form         # 原形（基本形）を取得

            # 記号や空白、サンプルの除外（Janomeでは「記号」という品詞になります）
            if pos in {"記号", "その他"}:
                continue
                
            if "助詞" in pos:
                particle_counter[surface] += 1

            tokens.append(surface)
            all_words.append(base)  # TTR用に原形を蓄積
            pos_counter[pos] += 1
            word_count += 1

            # 【部分一致判定】Janomeの品詞体系に引っかかるか安全にチェック
            is_function = any(f_pos in pos_full for f_pos in FUNCTION_POS)
            is_content = any(c_pos in pos_full for c_pos in CONTENT_POS)

            if is_function:
                function_words.append(surface)
                function_count += 1
            elif is_content:
                content_words.append(surface)
                content_count += 1

        # テキストの結合
        tokenized_texts.append(" ".join(tokens))
        function_texts.append(" ".join(function_words))
        content_texts.append(" ".join(content_words))
        function_counts.append(function_count)
        content_counts.append(content_count)

        # 割合計算
        if word_count > 0:
            f_per = (function_count / word_count) * 100
            c_per = (content_count / word_count) * 100
            comma_ratio = comma_count / word_count
            period_ratio = period_count / word_count
            ttr = len(set(all_words)) / word_count
        else:
            f_per = c_per = comma_ratio = period_ratio = ttr = 0

        function_pers.append(f_per)
        content_pers.append(c_per)

        total_particle = sum(particle_counter.values())
        particle_ratio = {
            f"助詞_{k}": (v / total_particle if total_particle > 0 else 0)
            for k, v in particle_counter.items()
        }
        pos_ratio = {
            pos_name: (count / word_count) * 100 for pos_name, count in pos_counter.items()
        }

        feature_data.append({
            "総単語数": word_count,
            "異なり語数": len(set(all_words)),
            "TTR": ttr,
            "平均文長": avg_sentence_length,
            "名詞割合": pos_ratio.get("名詞", 0),
            "動詞割合": pos_ratio.get("動詞", 0),
            "形容詞割合": pos_ratio.get("形容詞", 0),
            "助詞割合": pos_ratio.get("助詞", 0),
            "助動詞割合": pos_ratio.get("助動詞", 0),
            "副詞割合": pos_ratio.get("副詞", 0),
            **particle_ratio,
            "読点頻度": comma_ratio,
            "句点頻度": period_ratio,
            "感嘆符数": exclamation_count,
            "疑問符数": question_count,
        })

    return {
        "files": files,
        "tokenized_texts": tokenized_texts,
        "content_texts": content_texts,
        "function_texts": function_texts,
        "function_counts": function_counts,
        "content_counts": content_counts,
        "function_pers": function_pers,
        "content_pers": content_pers,
        "feature_data": feature_data,
    }

def jpn_word_ngram_analyzer(text):
    """半角スペースを挟まずに日本語の単語N-gramを生成する"""
    tokens = text.split()
    ngrams = []
    for n in [2, 3]:
        for i in range(len(tokens) - n + 1):
            ngrams.append("".join(tokens[i : i + n]))
    return ngrams

def print_analysis_results(data, X_content, content_features, Y_function, function_features, X_function_ngram, function_ngram_features, X_content_ngram, content_ngram_features, X_char_ngram, char_ngram_features, top_k=30):
    """解析結果を画面にわかりやすく表示する"""
    for i, (filepath, text) in enumerate(data["files"]):
        function_ngram_values = X_function_ngram.toarray()[i]
        content_ngram_values = X_content_ngram.toarray()[i]
        char_ngram_values = X_char_ngram.toarray()[i]

        function_ngram_top = dict(
            sorted(
                [(f, int(v)) for f, v in zip(function_ngram_features, function_ngram_values)],
                key=lambda x: x[1],
                reverse=True
            )[:top_k]
        )
        content_ngram_top = dict(
            sorted(
                [(f, int(v)) for f, v in zip(content_ngram_features, content_ngram_values)],
                key=lambda x: x[1],
                reverse=True
            )[:top_k]
        )
        char_top = dict(
            sorted(
                [(f, int(v)) for f, v in zip(char_ngram_features, char_ngram_values)], 
                key=lambda x: x[1], 
                reverse=True
            )[:top_k]
        )

        data["feature_data"][i].update({
            "function_ngram_top": function_ngram_top,
            "content_ngram_top": content_ngram_top,
            "char_ngram_top": char_top,
        })

        print(f"\n=== ファイル：{filepath} ===")
        for key, value in data["feature_data"][i].items():
            if isinstance(value, float):
                print(f"{key}：{value:.4f}")
            else:
                print(f"{key}：{value}")

        print("\n=== 機能語 ===")
        print(data["function_texts"][i])
        print(f"カウント数：{data['function_counts'][i]}個")
        print(f"割合：{data['function_pers'][i]:.2f}%")
        for word, score in zip(function_features, Y_function.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")

        print("\n=== 内容語 ===")
        print(f"カウント数：{data['content_counts'][i]}個")
        print(f"割合：{data['content_pers'][i]:.2f}%")
        for word, score in zip(content_features, X_content.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")


# -----------------------------------------------------------------
# メイン処理
# -----------------------------------------------------------------
analysis_data = analyze_text_files("Human_text")

if analysis_data["files"]:
    has_content = any(text.strip() for text in analysis_data["content_texts"])
    has_function = any(text.strip() for text in analysis_data["function_texts"])

    if not has_content or not has_function:
        print("\n【警告】抽出された内容語、または機能語が完全に空のファイルがあります。")
        exit()

    content_vectorizer = TfidfVectorizer()
    X_content = content_vectorizer.fit_transform(analysis_data["content_texts"])
    content_features = content_vectorizer.get_feature_names_out()

    function_vectorizer = TfidfVectorizer()
    Y_function = function_vectorizer.fit_transform(analysis_data["function_texts"])
    function_features = function_vectorizer.get_feature_names_out()

    function_ngram_vectorizer = CountVectorizer(analyzer=jpn_word_ngram_analyzer, max_features=2000)
    X_function_ngram = function_ngram_vectorizer.fit_transform(analysis_data["function_texts"])
    function_ngram_features = function_ngram_vectorizer.get_feature_names_out()

    content_ngram_vectorizer = CountVectorizer(analyzer=jpn_word_ngram_analyzer, max_features=2000)
    X_content_ngram = content_ngram_vectorizer.fit_transform(analysis_data["content_texts"])
    content_ngram_features = content_ngram_vectorizer.get_feature_names_out()

    raw_texts = [text for _, text in analysis_data["files"]]
    char_ngram_vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 5), max_features=3000)
    X_char_ngram = char_ngram_vectorizer.fit_transform(raw_texts)
    char_ngram_features = char_ngram_vectorizer.get_feature_names_out()

    print_analysis_results(
        analysis_data,
        X_content, content_features,
        Y_function, function_features,
        X_function_ngram, function_ngram_features,
        X_content_ngram, content_ngram_features,
        X_char_ngram, char_ngram_features
    )
else:
    print("処理対象のデータがありません。")


=== ファイル：/workspace/日本語文書のベクトル埋め込み/形態素解析器/Human_text_01.txt ===
総単語数：233
異なり語数：87
TTR：0.3734
平均文長：46.0000
名詞割合：40.3433
動詞割合：12.8755
形容詞割合：0.4292
助詞割合：33.0472
助動詞割合：12.4464
副詞割合：0.4292
助詞_は：0.1688
助詞_に：0.1039
助詞_て：0.0779
助詞_として：0.0519
助詞_か：0.0130
助詞_が：0.1688
助詞_で：0.1169
助詞_や：0.0260
助詞_を：0.1039
助詞_の：0.1429
助詞_と：0.0260
読点頻度：0.0687
句点頻度：0.0429
感嘆符数：0
疑問符数：0
function_ngram_top：{'では': 7, 'はの': 5, 'てます': 4, 'のを': 4, 'はのを': 4, 'がます': 3, 'がますで': 3, 'ではの': 3, 'なを': 3, 'ますで': 3, 'をが': 3, 'をに': 3, 'がで': 2, 'がでは': 2, 'たで': 2, 'てた': 2, 'としては': 2, 'にが': 2, 'にて': 2, 'にてます': 2, 'のをに': 2, 'ました': 2, 'ましたで': 2, 'ますが': 2, 'ますでは': 2, 'かとして': 1, 'かとしては': 1, 'がが': 1, 'ががます': 1, 'がた': 1}
content_ngram_top：{'スマートフォン': 7, 'いスマート': 3, 'いスマートフォン': 3, 'するの': 2, 'するの向い': 2, 'できること': 2, 'の向い': 2, 'の向いい': 2, 'コンテンツ生産': 2, '向いい': 2, '向いいスマート': 2, '現代社会': 2, 'SMS簡単': 1, 'SMS簡単ゲーム': 1, 'あっガラケー': 1, 'あっガラケーハイブリット': 1, 'ありガラケー': 1, 'ありガラケー出来': 1, 'ありパソコン': 1, 'ありパソコンコンテンツ': 1, 'いアプリ': 1, 'いアプリ使う': 1, 'い電話': 1, 'い電話カメラ': 1